# 05 · Promo Experiment Design (A/B Test)

Step 6 found that promo lift fades through the week (+65% Monday → +18% Friday). This notebook designs a store-level A/B test of a **shorter Mon–Wed promo** vs the current **Mon–Fri promo**, using historical data to size it.

1. Estimate the expected sales impact of dropping Thu–Fri promo
2. Define the metric, with a pre-period adjustment to cut noise
3. Validate the analysis with A/A tests (no false positives)
4. Simulate power to choose test duration and store count
5. Produce the store assignment and analysis plan

## 1. Load data

In [ ]:
import pandas as pd, numpy as np, pyodbc, json
import matplotlib.pyplot as plt
from scipy import stats

conn = pyodbc.connect("Driver={ODBC Driver 18 for SQL Server};Server=localhost;"
                      "Database=RetailForecast;Trusted_Connection=yes;TrustServerCertificate=yes;")
df = pd.read_sql("""
    SELECT d.Store, d.SalesDate AS [Date], d.DayOfWeek, d.Sales, d.Customers, d.Promo, c.Cluster
    FROM daily_sales d JOIN store_clusters c ON c.Store = d.Store
    WHERE d.IsOpen = 1 AND d.Sales > 0""", conn, parse_dates=['Date'])
df['Promo'] = df['Promo'].astype(int)
df['Week'] = df['Date'] - pd.to_timedelta(df['Date'].dt.dayofweek, unit='D')
print(df.shape)

## 2. Expected effect of a Mon–Wed promo

Dropping Thu–Fri promo removes the lift on those days. Expected weekly loss = Σ (day's share of promo-week sales) × lift / (1 + lift), using the matched weekday lifts from `04_promo_lift` section 4. This ignores any change in Saturday behavior.

In [ ]:
lift_by_day = {4: 0.282, 5: 0.176}          # Thu, Fri sales lift from 04_promo_lift

promo_weeks = df.groupby('Week')['Promo'].max()
pw = df[df.Week.isin(promo_weeks.index[promo_weeks == 1]) & (df.Date.dt.month != 12)]
day_share = pw.groupby('DayOfWeek')['Sales'].sum() / pw['Sales'].sum()
print(day_share.map('{:.1%}'.format).rename('share of promo-week sales'))

expected_drop = sum(day_share[d] * l / (1 + l) for d, l in lift_by_day.items())
print(f"\nExpected loss in promo-week sales from dropping Thu–Fri promo: {expected_drop:.1%}")

## 3. Store × promo-week outcome table

Unit of randomization = store. Outcome = a store's **average sales per open day** in promo weeks, so a holiday closure means one fewer day rather than a lost store-week. Store-weeks with more than 2 days below the store's normal open days (long closures) and December are excluded.

In [ ]:
sw = (df.groupby(['Store', 'Week'])
        .agg(Sales=('Sales', 'sum'), open_days=('Date', 'size'), Cluster=('Cluster', 'first'))
        .reset_index())
sw['SalesPerDay'] = sw['Sales'] / sw['open_days']
normal_days = sw.groupby('Store')['open_days'].agg(lambda s: s.mode().iloc[0])
sw = sw[sw.open_days >= sw.Store.map(normal_days) - 2]
sw = sw[sw.Week + pd.Timedelta(days=6) <= df.Date.max()]      # drop the partial final week
sw = sw[sw.Week.isin(promo_weeks.index[promo_weeks == 1]) & (sw.Week.dt.month != 12)]

pivot = sw.pivot(index='Store', columns='Week', values='SalesPerDay').sort_index(axis=1)
weeks = list(pivot.columns)
cluster = df.groupby('Store')['Cluster'].first()
print(f"{pivot.shape[0]} stores × {len(weeks)} eligible promo weeks")

## 4. Metric and variance reduction

Comparing raw weekly sales across stores is noisy because store size dominates. Instead, each store is compared with **itself**: outcome = log(avg daily sales in the test promo weeks ÷ avg daily sales in the 8 prior promo weeks). This is the same idea as CUPED. Extreme stores (level shifts from reopenings or data gaps) are capped at the 1st/99th percentile. The table shows how much this shrinks the noise.

In [ ]:
PRE = 8

def window(i, k):
    """Outcome per store for a test window starting at promo week i, lasting k promo weeks."""
    pre, post = pivot[weeks[i - PRE:i]], pivot[weeks[i:i + k]]
    ok = (pre.notna().sum(axis=1) >= PRE - 2) & (post.notna().sum(axis=1) >= max(1, k - 1))
    pre_m, post_m = pre.mean(axis=1)[ok], post.mean(axis=1)[ok]
    out = pd.DataFrame({'y_ratio': np.log(post_m / pre_m), 'y_raw': np.log(post_m)})
    return out.clip(out.quantile(0.01), out.quantile(0.99), axis=1)   # cap extreme stores (1st/99th pct)

K_LIST = [2, 4, 6]
MIN_COVER = 0.75                              # skip windows covering <75% of stores
windows = {}
for k in K_LIST:
    all_w = [window(i, k) for i in range(PRE, len(weeks) - k + 1)]
    windows[k] = [w for w in all_w if len(w) >= MIN_COVER * len(pivot)]
    print(f"k={k}: kept {len(windows[k])} of {len(all_w)} windows | "
          f"stores per window {min(len(w) for w in windows[k])}–{max(len(w) for w in windows[k])}")

var_tbl = pd.DataFrame({k: {'windows': len(w),
                            'sd_raw': np.median([x.y_raw.std() for x in w]),
                            'sd_ratio': np.median([x.y_ratio.std() for x in w])} for k, w in windows.items()}).T
var_tbl['noise_reduction'] = 1 - var_tbl.sd_ratio / var_tbl.sd_raw
print(var_tbl.round(3))

## 5. Stratified randomization

Stores are split within strata (cluster × sales-level quartile) so both groups get the same mix of store types and sizes.

In [ ]:
level = pivot.mean(axis=1)
strata = pd.Series(index=pivot.index, dtype=int)
for c, idx in cluster.loc[pivot.index].groupby(cluster.loc[pivot.index]).groups.items():
    q = min(4, max(1, len(idx) // 10))
    strata.loc[idx] = c * 10 + (pd.qcut(level.loc[idx], q, labels=False) if q > 1 else 0)
print(f"{strata.nunique()} strata")

def assign(stores, share, rng):
    s = strata.loc[stores]
    treat = pd.Series(False, index=stores)
    for _, idx in s.groupby(s).groups.items():
        idx = rng.permutation(np.array(idx))
        treat.loc[idx[:int(round(len(idx) * share))]] = True
    return treat

## 6. A/A test: does the analysis produce false positives?

Random splits on historical windows where nothing changed. About 5% of tests should come out significant at α = 0.05.

In [ ]:
rng = np.random.default_rng(42)

def run_test(w, share, effect, rng, col='y_ratio'):
    t = assign(w.index, share, rng)
    y = w[col] + np.where(t, np.log(1 - effect), 0)
    diff = y[t].mean() - y[~t].mean()
    p = stats.ttest_ind(y[t], y[~t], equal_var=False).pvalue
    return diff, p

aa = [run_test(windows[4][rng.integers(len(windows[4]))], 0.5, 0.0, rng) for _ in range(1000)]
aa_p = np.array([p for _, p in aa])
aa_d = np.array([d for d, _ in aa])
print(f"A/A false positive rate: {(aa_p < 0.05).mean():.1%} (target ≈ 5%)")
print(f"Mean estimated effect: {aa_d.mean() * 100:+.3f}% (target ≈ 0)")

## 7. Power simulation

Inject a true sales drop into the treatment group and count how often the test detects it.

In [ ]:
effects = [0.005, 0.01, 0.02, 0.03, 0.05]
designs = [(k, 0.5) for k in K_LIST] + [(2, 0.2), (4, 0.2)]
SIMS = 300
power = {}
for k, share in designs:
    for e in effects:
        hits = [run_test(windows[k][rng.integers(len(windows[k]))], share, e, rng)[1] < 0.05 for _ in range(SIMS)]
        power[(k, share, e)] = np.mean(hits)
pw_tbl = pd.Series(power).unstack()
pw_tbl.index = [f"{k} promo wks, {int(s * 100)}% stores treated" for k, s in pw_tbl.index]
pw_tbl.columns = [f"-{e * 100:g}%" for e in pw_tbl.columns]
print(pw_tbl.map('{:.0%}'.format))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for (label, row), mk in zip(pw_tbl.iterrows(), ['o', 's', '^', 'D', 'v']):
    ax.plot([e * 100 for e in effects], row.values * 100, marker=mk, label=label)
ax.axhline(80, color='grey', ls='--', lw=1)
ax.axvline(expected_drop * 100, color='#C44E52', ls=':', label=f'Expected drop ({expected_drop:.1%})')
ax.set(title='Power to detect a drop in promo-week sales', xlabel='True sales drop (%)', ylabel='Power (%)', ylim=(0, 105))
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('../figures/experiment_power.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Recommended design and store assignment

Pick the shortest design that detects a 2% drop with at least 80% power, preferring fewer treated stores.

In [ ]:
TARGET = 0.02
ok = [(k, s) for k, s in sorted(designs, key=lambda d: (d[0], d[1])) if power[(k, s, TARGET)] >= 0.8]
k_rec, share_rec = ok[0] if ok else (max(K_LIST), 0.5)
print(f"Recommended: {k_rec} promo weeks, {share_rec:.0%} of stores on Mon–Wed promo "
      f"(power at -2%: {power[(k_rec, share_rec, TARGET)]:.0%})")

# Final assignment: stores with a complete recent pre-period (last 8 promo weeks)
pre_last = pivot[weeks[-PRE:]]
eligible = pre_last.index[pre_last.notna().sum(axis=1) >= PRE - 2]
print(f"Eligible stores: {len(eligible)} of {len(pivot)}")
final = assign(eligible, share_rec, np.random.default_rng(2026))
assignment = pd.DataFrame({'Store': final.index, 'group': np.where(final, 'treatment_mon_wed', 'control_mon_fri'),
                           'Cluster': cluster.loc[final.index].values, 'stratum': strata.loc[final.index].values})

bal = assignment.assign(pre_sales=pivot[weeks[-PRE:]].mean(axis=1).loc[final.index].values)
print("\nBalance check (pre-period avg daily sales in promo weeks):")
print(bal.groupby('group')['pre_sales'].agg(['count', 'mean', 'median']).round(0))
print("\nStores by cluster:")
print(pd.crosstab(bal.Cluster, bal.group))

## 9. Analysis plan (fixed before the test)

| Item | Plan |
|---|---|
| Hypothesis | A Mon–Wed promo loses less promo-week sales than the cost saved by cutting Thu–Fri promo |
| Unit | Store, stratified by cluster × sales-level quartile |
| Primary metric | log(avg daily sales in test promo weeks ÷ avg of prior 8 promo weeks) |
| Timing | Avoid Easter, the May–June holiday cluster and December; pre-period must also avoid heavy holiday weeks |
| Guardrails | Customer traffic, Saturday sales, non-promo-week sales |
| Test | Welch t-test on the primary metric (capped at 1st/99th percentile), α = 0.05, two-sided, 95% CI on the % change |
| Decision | Adopt Mon–Wed if the upper bound of the sales loss is below the promo cost saving (needs margin and promo cost from finance) |
| Exclusions | December, holiday and closure weeks, stores with incomplete pre-period |

## 10. Save outputs

In [ ]:
assignment.to_csv('../data/processed/experiment_assignment.csv', index=False)
summary = {
    'expected_drop': round(float(expected_drop), 4),
    'noise_reduction_4wk': round(float(var_tbl.loc[4, 'noise_reduction']), 4),
    'aa_false_positive_rate': round(float((aa_p < 0.05).mean()), 4),
    'recommended_promo_weeks': int(k_rec),
    'recommended_treated_share': float(share_rec),
    'power_at_2pct': round(float(power[(k_rec, share_rec, TARGET)]), 4),
    'treated_stores': int(final.sum()), 'control_stores': int((~final).sum()),
}
with open('../data/processed/experiment_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))